# LangChain Expression Language (LCEL) 🔗

## What is LCEL?
**LangChain Expression Language (LCEL)** is a declarative way to easily compose chains together. It acts as the fundamental syntax in modern LangChain, allowing you to string together components (like prompts, models, and output parsers) using a Unix-style pipe operator (`|`).

Instead of writing complex Python classes or deeply nested function calls to combine LLM components, you just "pipe" the output of one component directly into the input of the next.

---

## Why Use LCEL?
1. **First-class Streaming Support:** If you build your chain with LCEL, you get best-in-class time-to-first-token streaming automatically.
2. **Unified Interface:** Every LCEL chain implements the `Runnable` interface. This means any chain you build automatically supports `.invoke()`, `.stream()`, `.batch()`, and their async equivalents (`.ainvoke()`, etc.) right out of the box!
3. **Automatic Parallelism:** If you have components that can run at the same time, LCEL automatically executes them in parallel to save time.
4. **Built-in Fallbacks:** You can easily attach fallbacks (e.g., if OpenAI fails, fallback to a local Groq model) with `.with_fallbacks()`.

---

## The Syntax: The Pipe Operator (`|`)
The core of LCEL is the `|` symbol. It takes the output of the component on the left and passes it as the input to the component on the right.

```python
chain = component_1 | component_2 | component_3
```

## Example 1: The Basic Chain (Prompt | Model | Parser)
This is the most common pattern in LangChain. We take user input, format it into a prompt, pass it to the model, and then extract the text from the model's message object.

In [3]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain.chat_models import init_chat_model

# 1. Create a prompt template
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant that speaks like a pirate."),
    ("user", "Tell me about {topic}")
])

# 2. Initialize the model
model = init_chat_model("llama-3.1-8b-instant", model_provider="groq")

# 3. Initialize an output parser (Extracts the string from the AIMessage object)
parser = StrOutputParser()

# --- Build the LCEL Chain ---
chain = prompt | model | parser

# Run the chain! 
# We pass a dictionary matching the {topic} variable in the prompt.
response = chain.invoke({"topic": "artificial intelligence"})
print(response)

Ye be lookin' fer a tale of artificial intelligence, eh? Alright then, settle yerself down with a pint o' grog and listen close, for I be tellin' ye a story o' machines that think like the scurvy dogs themselves.

Artificial intelligence, or AI, be the study o' creatin' machines that can think, learn, and act like livin' beings. It be a field that's been makin' waves in the tech world fer decades, and it's gettin' more sophisticated by the day.

There be several types o' AI, but I'll give ye the lowdown on the main ones:

1. **Narrow or Weak AI**: This be the kind o' AI that's designed to do one specific task, like playin' chess or recognizein' faces. It be like a trusty parrot that can squawk out a few phrases, but can't talk like a proper pirate.
2. **General or Strong AI**: This be the Holy Grail o' AI, matey. It be the kind o' AI that can think and learn like a human, and can do just about any task that a human can do. It be like havin' a crew o' elves that can do all the work fer 

**How it flows:** 
`{"topic": "artificial intelligence"}` ➡️ `prompt` ➡️ `Message Object` ➡️ `model` ➡️ `AIMessage` ➡️ `parser` ➡️ `String`

## Example 2: RunnablePassthrough (Injecting Data)
Often, you want to pass data directly through without altering it, or add new keys to your input dictionary before it hits the prompt. You use `RunnablePassthrough` for this.

In [ ]:
from langchain_core.runnables import RunnablePassthrough

# Imagine we have a function that retrieves context from a database
def get_context(query: str):
    return "LangChain is a framework for developing applications powered by language models."

# We build a chain that first fetches context, adds it to the dictionary, and passes it to the prompt.
chain = (
    {"context": get_context, "topic": RunnablePassthrough()} 
    | prompt 
    | model 
    | parser
)

# Because we used RunnablePassthrough for the 'topic', the string we pass to invoke() 
# becomes the value for 'topic'.
chain.invoke("LangChain")

## Example 3: Parallel Execution (RunnableParallel)
Sometimes you want to format multiple things at the same time. LCEL handles this automatically when you use a dictionary.

In [ ]:
from langchain_core.runnables import RunnableParallel

chain1 = ChatPromptTemplate.from_template("Tell me a joke about {topic}") | model | parser
chain2 = ChatPromptTemplate.from_template("Write a poem about {topic}") | model | parser

# Run both chains AT THE SAME TIME
combined_chain = RunnableParallel(
    joke=chain1,
    poem=chain2
)

# This will take only as long as the slowest chain, rather than the sum of both!
result = combined_chain.invoke({"topic": "bears"})
print("Joke:", result["joke"])
print("Poem:", result["poem"])

## Summary
Whenever you are building with LangChain, always think: **"Can I express this as an LCEL pipeline using `|`?"** 
It drastically reduces boilerplate code, makes your logic highly readable, and gives you enterprise-grade features (like streaming and async) for free.